# Classification Track - Review 2 Part B

**Problem:** Predict whether a room is occupied (`Occupancy` = 0 or 1) from environmental sensors.

**Dataset:** UCI Occupancy Detection. This notebook continues `notebooks/classification.ipynb` (Review 1 Part A): same files, same CO2 filter, same features, and the same official train/test split.

**Part A, refit with the hyperparameters already chosen:** Logistic Regression, K-Nearest Neighbors, Gaussian Naive Bayes, Decision Tree, Support Vector Machine.

**Part B, trained here:** Random Forest, AdaBoost, Gradient Boosting, Bagging, MLP.

**Metrics for every model:** Accuracy, Precision (weighted), Recall (weighted), F1 (weighted), ROC-AUC, and a confusion matrix. The last table holds all ten classifiers.


In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_theme(palette="colorblind")
RANDOM_STATE = 42


## 1. Same room-occupancy split as Part A

Training file: `data/datatraining.txt`. Test files: `data/datatest.txt` and `data/datatest2.txt`, concatenated. This is the UCI official split used in Review 1, so Part B is compared on the same rows as Part A.

CO2 readings above 2000 are removed as sensor anomalies, using the same rule as Part A. Features are Temperature, Light, CO2, HumidityRatio, seconds since midnight (NSM), Weekday, and Weekend. Raw Humidity is left out because HumidityRatio is already the temperature-adjusted humidity supplied with the dataset.


In [2]:
def clean_data(df):
    # Same anomaly rule as Review 1 Part A.
    return df.query("CO2 <= 2000").reset_index(drop=True)


def prepare_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["NSM"] = (
        df["date"].dt.hour * 3600
        + df["date"].dt.minute * 60
        + df["date"].dt.second
    )
    # Monday-Friday vs Saturday-Sunday. Weekend is the complement of Weekday.
    df["Weekday"] = df["date"].dt.dayofweek < 5
    df["Weekend"] = df["date"].dt.dayofweek >= 5
    return df


FEATURES = [
    "Temperature",
    "Light",
    "CO2",
    "HumidityRatio",
    "NSM",
    "Weekday",
    "Weekend",
]
TARGET = "Occupancy"

train = prepare_features(clean_data(pd.read_csv("../data/datatraining.txt")))
test = prepare_features(clean_data(pd.concat([
    pd.read_csv("../data/datatest.txt"),
    pd.read_csv("../data/datatest2.txt"),
], ignore_index=True)))

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Train occupied rate: {:.4f}".format(y_train.mean()))
print("Test occupied rate: {:.4f}".format(y_test.mean()))
print("Missing values:", int(pd.concat([X_train, X_test]).isna().sum().sum()))


Training set: (8102, 7)
Testing set: (12414, 7)
Train occupied rate: 0.2083
Test occupied rate: 0.2434
Missing values: 0


### Split observation

**Observation:** After the CO2 filter, training has **8,102** rows and the official test files have **12,414**. Occupied rows are the minority: **20.83%** of training rows and **24.34%** of test rows. The seven model features have no missing values. The test file is larger than the training file because that is the published UCI split; Part B keeps it so every model is scored on the same rows as Part A.


## 2. Part A models, refit with the Review 1 hyperparameters

These five settings won the Part A grid search (5-fold weighted F1). They are refit here, not searched again, so precision, recall, and ROC-AUC are computed with the same function as the Part B models.

| Model | Hyperparameters from Part A |
|---|---|
| Logistic Regression | C=10, solver=lbfgs |
| K-Nearest Neighbors | k=19, Manhattan distance |
| Gaussian Naive Bayes | var_smoothing=1e-4 |
| Decision Tree | criterion=entropy, max_depth=3 |
| Support Vector Machine | C=0.1, kernel=linear |

Scaling matches Part A: RobustScaler inside the pipeline for every model except the decision tree. The scaler is fit on the training fold only.


In [3]:
def positive_scores(model, X):
    # The linear SVM matches Part A and was fit without class probabilities.
    # ROC-AUC accepts those decision scores. Every other model uses P(occupied).
    final = model.named_steps["clf"]
    if isinstance(final, SVC):
        return model.decision_function(X)
    return model.predict_proba(X)[:, 1]


def metric_row(name, model, track):
    y_pred = model.predict(X_test)
    return {
        "Track": track,
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, positive_scores(model, X_test)),
    }


part_a_models = {
    "Logistic Regression": Pipeline([
        ("scaler", RobustScaler()),
        ("clf", LogisticRegression(C=10.0, solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", RobustScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=19, metric="manhattan")),
    ]),
    "Naive Bayes (Gaussian)": Pipeline([
        ("scaler", RobustScaler()),
        ("clf", GaussianNB(var_smoothing=1e-4)),
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=RANDOM_STATE)),
    ]),
    "Support Vector Machine": Pipeline([
        ("scaler", RobustScaler()),
        ("clf", SVC(C=0.1, kernel="linear", random_state=RANDOM_STATE)),
    ]),
}

for name, model in part_a_models.items():
    model.fit(X_train, y_train)
    row = metric_row(name, model, "A")
    print("{:28s}  Acc {:.4f}  F1 {:.4f}  ROC-AUC {:.4f}".format(
        name, row["Accuracy"], row["F1"], row["ROC-AUC"]
    ))


Logistic Regression           Acc 0.9904  F1 0.9905  ROC-AUC 0.9951


K-Nearest Neighbors           Acc 0.9741  F1 0.9741  ROC-AUC 0.9927
Naive Bayes (Gaussian)        Acc 0.9820  F1 0.9822  ROC-AUC 0.9957
Decision Tree                 Acc 0.9385  F1 0.9377  ROC-AUC 0.9907
Support Vector Machine        Acc 0.9877  F1 0.9878  ROC-AUC 0.9953


### Part A refit observation

**Observation:** These refits match the Review 1 test scores: Logistic Regression **0.9904** accuracy and **0.9905** weighted F1, SVM **0.9877 / 0.9878**, Gaussian NB **0.9820 / 0.9822**, KNN **0.9741 / 0.9741**, Decision Tree **0.9385 / 0.9377**. ROC-AUC is above **0.99** for all five, including the decision tree (**0.9907**). The tree ranks occupied intervals well and then loses accuracy at the classification threshold.


## 3. Part B models

Each Part B model is fit twice on the same training rows: a default baseline, then `GridSearchCV` with 5-fold weighted F1. Test metrics of the baseline and the selected model are reported side by side, because a higher cross-validation score does not always raise the test score.

Tree ensembles are left unscaled, matching the Part A decision tree. The MLP uses `RobustScaler` inside the pipeline.

| Model | What is tuned |
|---|---|
| Random Forest | `n_estimators`, `max_depth` |
| AdaBoost | `n_estimators`, `learning_rate` (decision-stump base) |
| Gradient Boosting | `n_estimators`, `learning_rate`, `max_depth` (sklearn GBM) |
| Bagging | `n_estimators`, tree `max_depth` (decision-tree base) |
| MLP | `hidden_layer_sizes`, `activation` |


In [4]:
def make_rf():
    return Pipeline([
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    ])


def make_ada():
    return Pipeline([
        ("scaler", RobustScaler()),
        ("clf", AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            random_state=RANDOM_STATE,
        )),
    ])


def make_gbm():
    return Pipeline([
        ("scaler", RobustScaler()),
        ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])


def make_bag():
    return Pipeline([
        ("clf", BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])


def make_mlp():
    return Pipeline([
        ("scaler", RobustScaler()),
        ("clf", MLPClassifier(max_iter=400, early_stopping=True, random_state=RANDOM_STATE)),
    ])


# Random Forest and Bagging stay unscaled, matching the Part A decision tree.
# MLP needs scaling. AdaBoost and Gradient Boosting use the same RobustScaler
# pipeline so their inputs match the scaled Part A models.

builders = {
    "Random Forest": make_rf,
    "AdaBoost": make_ada,
    "Gradient Boosting": make_gbm,
    "Bagging": make_bag,
    "MLP": make_mlp,
}

param_grids = {
    "Random Forest": {
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 12, 20],
    },
    "AdaBoost": {
        "clf__n_estimators": [50, 100, 200],
        "clf__learning_rate": [0.05, 0.1, 0.5, 1.0],
    },
    "Gradient Boosting": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.05, 0.1, 0.2],
        "clf__max_depth": [2, 3],
    },
    "Bagging": {
        "clf__n_estimators": [20, 50, 100],
        "clf__estimator__max_depth": [3, 5, None],
    },
    "MLP": {
        "clf__hidden_layer_sizes": [(32,), (64,), (64, 32), (100, 50)],
        "clf__activation": ["relu", "tanh"],
    },
}

baseline_models = {}
tuned_models = {}
search_results = {}

for name, builder in builders.items():
    print("Fitting", name)
    baseline = builder()
    baseline.fit(X_train, y_train)
    baseline_models[name] = baseline

    search = GridSearchCV(
        estimator=builder(),
        param_grid=param_grids[name],
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    search_results[name] = search
    print("  best params:", search.best_params_)
    print("  best CV weighted F1: {:.4f}".format(search.best_score_))


Fitting Random Forest


  best params: {'clf__max_depth': 12, 'clf__n_estimators': 100}
  best CV weighted F1: 0.9669
Fitting AdaBoost


  best params: {'clf__learning_rate': 0.5, 'clf__n_estimators': 50}
  best CV weighted F1: 0.9860
Fitting Gradient Boosting


  best params: {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__n_estimators': 100}
  best CV weighted F1: 0.9714
Fitting Bagging


  best params: {'clf__estimator__max_depth': 5, 'clf__n_estimators': 100}
  best CV weighted F1: 0.9517
Fitting MLP


  best params: {'clf__activation': 'relu', 'clf__hidden_layer_sizes': (64, 32)}
  best CV weighted F1: 0.9869
